## Notebook content

This notebook submits a new AutoML **timeseries** pipeline run, monitors progress, and displays the model leaderboard when training completes.

💡 **Tips:**
- Ensure the AutoML pipeline is uploaded to your Kubeflow Pipelines instance before submitting.
- Configure training data and pipeline parameters before submitting.
- Attach an S3 connection to the workbench so artifact download works after the run completes.

### Contents

**[Setup](#setup)**  
**[Configuration](#configuration)**  
**[Connect to Kubeflow Pipelines](#connect-to-kubeflow-pipelines)**  
**[Submit pipeline run](#submit-pipeline-run)**  
**[Monitor run status](#monitor-run-status)**  
**[Retrieve run artifacts](#retrieve-run-artifacts)**  
**[Leaderboard](#leaderboard)**  
**[Summary and next steps](#summary-and-next-steps)**


<a id="setup"></a>
## Setup


In [ ]:
import warnings

warnings.filterwarnings("ignore")


In [ ]:
%pip install "kfp>=2.16" boto3 pandas | tail -n 1


<a id="configuration"></a>
## Configuration

Set data locations and pipeline parameters before submitting a new run.


In [ ]:
import os
from datetime import datetime, timezone

pipeline_name = "autogluon-timeseries-training-pipeline"

# Kubeflow Pipelines API
kfp_host = os.environ.get("KFP_HOST", "https://<REPLACE_KFP_HOST>/pipeline")
kfp_namespace = os.environ.get("KFP_NAMESPACE", "<REPLACE_NAMESPACE>")
kfp_token = os.environ.get("KFP_TOKEN")
kfp_verify_ssl = os.environ.get("KFP_VERIFY_SSL", "true").lower() not in {"0", "false", "no"}

# Artifact store bucket (pipeline outputs are written under <pipeline_name>/<run_id>/)
artifacts_bucket = os.environ.get("AWS_S3_BUCKET", "<REPLACE_ARTIFACTS_BUCKET>")

# Training data
train_data_secret_name = "<REPLACE_S3_SECRET>"
train_data_bucket_name = "<REPLACE_DATA_BUCKET>"
train_data_file_key = "<REPLACE_DATA_FILE_KEY>"

# Optional user-provided test dataset (leave both empty for internal holdout split)
test_data_bucket_name = "<REPLACE_TEST_DATA_BUCKET>"
test_data_file_key = "<REPLACE_TEST_DATA_FILE_KEY>"

# Pipeline parameters
target = "<REPLACE_TARGET>"
id_column = "<REPLACE_ID_COLUMN>"
timestamp_column = "<REPLACE_TIMESTAMP_COLUMN>"
known_covariates_names = <REPLACE_KNOWN_COVARIATES_NAMES>
prediction_length = <REPLACE_PREDICTION_LENGTH>
top_n = <REPLACE_TOP_N>
eval_metric = "<REPLACE_EVAL_METRIC>"
preset = "<REPLACE_PRESET>"
experiment_name = os.environ.get("KFP_EXPERIMENT_NAME", "automl-experiments")

# Optional: pin a specific uploaded pipeline version (leave empty for default)
pipeline_version_id = ""

run_timeout_seconds = int(os.environ.get("KFP_RUN_TIMEOUT", "3600"))
poll_interval_seconds = 30

<a id="connect-to-kubeflow-pipelines"></a>
## Connect to Kubeflow Pipelines


In [ ]:
import kfp

client_kwargs = {
    "host": kfp_host if kfp_host.endswith("/") else f"{kfp_host}/",
    "namespace": kfp_namespace,
    "verify_ssl": kfp_verify_ssl,
}
if kfp_token:
    client_kwargs["existing_token"] = kfp_token

client = kfp.Client(**client_kwargs)
print(f"Connected to KFP in namespace: {client.get_user_namespace()}")


<a id="submit-pipeline-run"></a>
## Submit pipeline run

Submit a new run using the pipeline already uploaded to Kubeflow Pipelines.


In [ ]:
pipeline_arguments = {
    "train_data_secret_name": train_data_secret_name,
    "train_data_bucket_name": train_data_bucket_name,
    "train_data_file_key": train_data_file_key,
    "test_data_bucket_name": test_data_bucket_name,
    "test_data_file_key": test_data_file_key,
    "target": target,
    "id_column": id_column,
    "timestamp_column": timestamp_column,
    "known_covariates_names": known_covariates_names,
    "prediction_length": prediction_length,
    "top_n": top_n,
    "eval_metric": eval_metric,
    "preset": preset,
}

pipeline_id = client.get_pipeline_id(pipeline_name)
if not pipeline_id:
    raise RuntimeError(
        f"Pipeline '{pipeline_name}' was not found in KFP. "
        "Upload the pipeline to this cluster before running this notebook."
    )

experiment = client.create_experiment(name=experiment_name)
run_name = f"automl-timeseries-experiment-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}"

run_kwargs = {
    "experiment_id": experiment.experiment_id,
    "job_name": run_name,
    "pipeline_id": pipeline_id,
    "params": pipeline_arguments,
}
if pipeline_version_id:
    run_kwargs["version_id"] = pipeline_version_id

run_info = client.run_pipeline(**run_kwargs)
run_id = run_info.run_id
print(f"Submitted run: {run_name}")
print(f"Run ID: {run_id}")
print(f"Pipeline: {pipeline_name}")
if pipeline_version_id:
    print(f"Pipeline version: {pipeline_version_id}")

<a id="monitor-run-status"></a>
## Monitor run status

Poll the run until it reaches a terminal state.


In [ ]:
import time


def _run_state(run_detail) -> str:
    run = getattr(run_detail, "run", run_detail)
    state = getattr(run, "state", None)
    if state is None and hasattr(run, "status"):
        state = getattr(run.status, "state", None)
    return str(state or "UNKNOWN").upper()


def poll_run_status(client, run_id: str, timeout_seconds: int, interval_seconds: int = 30):
    terminal = {"SUCCEEDED", "FAILED", "CANCELED", "CANCELLED", "SKIPPED"}
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        detail = client.get_run(run_id)
        state = _run_state(detail)
        created = getattr(getattr(detail, "run", detail), "created_at", "")
        finished = getattr(getattr(detail, "run", detail), "finished_at", "")
        print(f"[{datetime.now(timezone.utc).isoformat()}] state={state} created={created} finished={finished}")
        if state in terminal:
            return detail, state
        time.sleep(interval_seconds)
    raise TimeoutError(f"Run {run_id} did not finish within {timeout_seconds}s")


run_detail, final_state = poll_run_status(
    client,
    run_id,
    timeout_seconds=run_timeout_seconds,
    interval_seconds=poll_interval_seconds,
)
print(f"Final state: {final_state}")
if final_state != "SUCCEEDED":
    raise RuntimeError(f"Pipeline run {run_id} finished with state {final_state}")


<a id="retrieve-run-artifacts"></a>
## Retrieve run artifacts

Download run-level artifacts from `s3://<artifacts_bucket>/<pipeline_name>/<run_id>/`.


In [ ]:
import json
from pathlib import Path

import boto3
import pandas as pd
from botocore.exceptions import SSLError

required_env_vars = (
    "AWS_S3_ENDPOINT",
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
)
missing_vars = [name for name in required_env_vars if not os.environ.get(name, "").strip()]
if missing_vars:
    raise ValueError(
        f"Missing required environment variable(s): {', '.join(missing_vars)}. "
        "Attach your S3 connection to this workbench and try again."
    )


def get_s3_client(verify: bool = True):
    return boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
        verify=verify,
    )


def list_run_object_keys(bucket: str, prefix: str, verify: bool = True) -> list[str]:
    keys: list[str] = []
    paginator = get_s3_client(verify=verify).get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents") or []:
            key = obj["Key"]
            if not key.endswith("/"):
                keys.append(key)
    return sorted(keys)


def download_keys(bucket: str, prefix: str, keys: list[str], dest_dir: Path, verify: bool = True) -> list[Path]:
    s3 = get_s3_client(verify=verify)
    downloaded: list[Path] = []
    for key in keys:
        relative = key.split(prefix, 1)[-1]
        target = dest_dir / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(bucket, key, str(target))
        downloaded.append(target)
    return downloaded


artifact_prefix = f"{pipeline_name}/{run_id}/"
local_output_dir = Path("automl_run_artifacts") / run_id
local_output_dir.mkdir(parents=True, exist_ok=True)

try:
    artifact_keys = list_run_object_keys(artifacts_bucket, artifact_prefix)
except SSLError:
    warnings.warn("SSL error when listing S3 objects, retrying with verify=False")
    artifact_keys = list_run_object_keys(artifacts_bucket, artifact_prefix, verify=False)

interesting_suffixes = (
    "component_status.json",
    "html_artifact",
    "model.json",
)
selected_keys = [
    key
    for key in artifact_keys
    if any(key.endswith(suffix) for suffix in interesting_suffixes)
]

try:
    downloaded_paths = download_keys(artifacts_bucket, artifact_prefix, selected_keys, local_output_dir)
except SSLError:
    warnings.warn("SSL error when downloading artifacts, retrying with verify=False")
    downloaded_paths = download_keys(artifacts_bucket, artifact_prefix, selected_keys, local_output_dir, verify=False)

print(f"Downloaded {len(downloaded_paths)} file(s) to {local_output_dir.resolve()}")


<a id="leaderboard"></a>
## Leaderboard

Review ranked refitted models from this run.


In [ ]:
from IPython.display import HTML, display

leaderboard_paths = sorted(local_output_dir.rglob("html_artifact"))
if not leaderboard_paths:
    raise FileNotFoundError(
        f"Leaderboard artifact not found under {local_output_dir}. "
        "Ensure the run succeeded and the S3 connection can access pipeline outputs."
    )

leaderboard_html = leaderboard_paths[0].read_text(encoding="utf-8")
display(HTML(leaderboard_html))

model_rows = []
for model_json_path in sorted(local_output_dir.rglob("model.json")):
    with model_json_path.open(encoding="utf-8") as f:
        model_doc = json.load(f)
    metrics = (model_doc.get("metrics") or {}).get("test_data") or {}
    row = {"model": model_doc.get("name"), **metrics}
    row["predictor_notebook"] = (model_doc.get("location") or {}).get("notebook")
    model_rows.append(row)

if model_rows:
    display(pd.DataFrame(model_rows))
else:
    print("No per-model metadata found.")


<a id="summary-and-next-steps"></a>
## Summary and next steps

You submitted a new AutoML timeseries pipeline run and reviewed its leaderboard.

**Next steps:**
1. Pick a model from the leaderboard table.
2. Open its predictor notebook from `models_artifact/<ModelName>_FULL/notebooks/automl_predictor_notebook.ipynb` for inference and deeper model analysis.
